# Cache-Augmented Generation (Cache-AG)

This notebook demonstrates a minimal implementation of **cache-augmented generation** (Cache-AG).

In this pattern, we pre-process a static context (like a large document or a knowledge base) once and store its computed state in the **Key-Value (KV) cache** of the Transformer model. Subsequent user queries can then reuse this cache, significantly reducing latency and compute costs.

## 1. Setup

We use the Hugging Face `transformers` library and a small model like `gpt2` for this demonstration.

In [ ]:
%pip install agno qdrant-client ollama pypdf fastapi uvicorn

In [1]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Optional, Tuple

# A type hint for the KV cache structure.
PastKeyValues = Tuple[Tuple[torch.Tensor, torch.Tensor], ...]

class TrueCacheAugmentedGenerator:
    def __init__(self, model_name: str = "gpt2"):
        print(f"Loading model: {model_name}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Running on device: {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Internal state for the cache
        self.kv_cache: Optional[PastKeyValues] = None

    def preload_context(self, context: str):
        """
        Phase 1: "Remembering"
        Processes the context and stores its computed state in the KV cache.
        """
        print(f"\n--- [Phase 1] Pre-loading context... ---")
        start_time = time.time()

        inputs = self.tokenizer(context, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, use_cache=True)

        self.kv_cache = outputs.past_key_values

        elapsed = (time.time() - start_time) * 1000
        print(f"Context ingested and cached in {elapsed:.2f}ms.")

    def ask(self, query: str, max_new_tokens: int = 50) -> str:
        """
        Phase 2: "Querying" using a manual generation loop.
        Reuses the pre-computed cache.
        """
        if self.kv_cache is None:
            raise ValueError("You must call `preload_context` before asking a question.")

        print(f"\n--- [Phase 2] Answering query: '{query}' ---")
        start_time = time.time()

        input_ids = self.tokenizer(query, return_tensors="pt").to(self.device).input_ids
        current_cache = self.kv_cache
        generated_token_ids = []

        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = self.model(
                    input_ids=input_ids,
                    past_key_values=current_cache,
                    use_cache=True
                )

            next_token_logits = outputs.logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
            generated_token_ids.append(next_token_id.item())

            if next_token_id.item() == self.tokenizer.eos_token_id:
                break

            current_cache = outputs.past_key_values
            input_ids = next_token_id

        elapsed = (time.time() - start_time) * 1000
        print(f"Query processed in {elapsed:.2f}ms.")

        return self.tokenizer.decode(generated_token_ids, skip_special_tokens=True)

generator = TrueCacheAugmentedGenerator(model_name="gpt2")

Loading model: gpt2...
Running on device: cpu


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 2. Ingest Context

We provide some facts to the model and cache them.

In [2]:
science_facts = (
    "Jupiter is the largest planet in our solar system. Its mass is "
    "two and a half times that of all the other planets in the Solar System combined. "
    "Mitochondria are membrane-bound cell organelles that generate most of "
    "the chemical energy needed to power the cell's biochemical reactions."
)
generator.preload_context(science_facts)


--- [Phase 1] Pre-loading context... ---
Context ingested and cached in 462.28ms.


## 3. Query using the Cache

We ask questions that refer to the pre-loaded context. Notice that the model doesn't need to re-process the facts for each question.

In [3]:
questions = [
    "What is the function of mitochondria?",
    "How massive is Jupiter?",
]

for q in questions:
    answer = generator.ask(q)
    print(f"  - User: {q}")
    print(f"  - AI: {answer}")


--- [Phase 2] Answering query: 'What is the function of mitochondria?' ---
Query processed in 3315.81ms.
  - User: What is the function of mitochondria?
  - AI:  The mitochondria are the most important organelles in the body. They are responsible for the production of energy, which is why they are called "energy-producing" cells. The mitochondria are also responsible for the production of oxygen, which is

--- [Phase 2] Answering query: 'How massive is Jupiter?' ---
Query processed in 2942.49ms.
  - User: How massive is Jupiter?
  - AI:  The largest planet in our solar system is about the size of Jupiter. It is about the size of the Earth. It is about the size of the Sun. It is about the size of the Moon. It is about the size of the Earth.


### Demonstrate Cache Reusability

We can ask a new question, and it will still be fast because the initial context is still in the cache.

In [4]:
new_question = "Which planet is the most massive in our solar system?"
answer = generator.ask(new_question)
print(f"  - User: {new_question}")
print(f"  - AI: {answer}")


--- [Phase 2] Answering query: 'Which planet is the most massive in our solar system?' ---
Query processed in 3579.56ms.
  - User: Which planet is the most massive in our solar system?
  - AI:  The Earth is about the size of the Sun. It is about the size of the Moon. It is about the size of the Earth. It is about the size of the Sun. It is about the size of the Earth. It is about the
